#### **Introduction to Importance Sampling and Metropolis-Hastings Monte Carlo** <br> Joshua Pajak, Ph.D. joshua.pajak@umassmed.edu

This notebook is designed to introduce the reader to key concepts underlying **importance sampling** using a **Markov Chain** method, specifically the **Metropolis-Hastings Monte Carlo** method to simulate a simple harmonic oscillator. 

Initially, we will sample the ensemble by generating configurations randomly and accepting them with the Metropolis-Hastings criteria. Then, we will implement a importance sampling approach to generating poses. We will see how this approach lets us approximate the ensemble much more quickly and efficiently.

In [ ]:
# --- First we import all the libraries we need ---
import numpy as np  # fast linear algebra
import pandas as pd # dataframes are good
import matplotlib.pyplot as plt # plotting
import seaborn as sns # more plotting
sns.set_theme('notebook')

Let's calculate the theoretical probability distribution functions (PDFs) for a harmonic oscillator in 1D so that we have something to vet our simulations against.

In [ ]:
# --- Make energy function so we can easily calculate energies of states ---
def total_energy_physical(x, v, k, x_center=0.0):
    """
    Calculates the total energy (Hamiltonian) of a ball on a spring.
    Args:
        x (float): position coordinate
        v (float): velocity coordinate
        k (float): spring constant
        x_center (float): the center of the spring (default=0.0)
    Returns:
        float: total energy E
    """
    E_potential = 0.5 * k * (x - x_center)**2
    E_kinetic = 0.5 * v**2
    return E_potential + E_kinetic

In [ ]:
def calculate_theoretical_distribution(k, KT, x_center=0.0, x_lim=10, v_lim=10, resolution=100):
    """
    Generates a DataFrame of theoretical probabilities for the harmonic oscillator
    across a grid of position and velocity values.
    Args:
        k (float): spring constant (reduced units; must be positive)
        KT (float): temperature factor (must be positive)
        x_center (float): center of spring (default=0.0)
        x_lim (float): limit to calculate position (default=10)
        v_lim (float): limit to calculate velocity (default=10)
        resolution (int): how many points in our meshgrid
        
    Returns:
        DataFrame: ['position', 'velocity', 'probability']
    """
    # --- Create a grid of x and v values --
    x_range = np.linspace(x_center - x_lim, x_center + x_lim, resolution)
    v_range = np.linspace(-v_lim, v_lim, resolution)
    X, V = np.meshgrid(x_range, v_range)

    # --- Calculate energy of every point in our meshgrid ---
    Energy = total_energy_physical(X,V,k,x_center)

    # --- Calculate the Boltzmann distribution ---
    prob_density = np.exp(-Energy / KT)

    # --- Normalize by the parition function --- 
    Z = (2 * np.pi * KT) / np.sqrt(k)
    prob_density /= Z

    # 5. Flatten and convert to a tidy DataFrame
    df_theory = pd.DataFrame({
        'position': X.ravel(),
        'velocity': V.ravel(),
        'probability': prob_density.ravel()
    })

    return df_theory

Great! So now will have a sense for what we are after. But now let's imagine we didn't know how to calculate the partition function, so we didn't know how to normalize our distribution. This is where **Metropolis-Hastings Monte Carlo** comes in handy. We will generate new states and **accept** or **reject** those states based on the **energy relative to our current state.** Because we are looking at the _ratio_ of states, the partition function cancels and we can approximate the probability distribution with the correct normalization. 

In [ ]:
# --- Define our acceptance criteria --- 
def acceptance_probability(delta_energy, KT):
    return min(1.0, np.exp(-delta_energy / KT))

First, we will implement a "naive" Metropolis-Hastings approach, where new states are generated truly randomly. To limit our search, we will set bounds $x_{lim}$ and $v_{lim}$, otherwise our search would have to be infinitely broad.  

In [ ]:
def metropolis_naive(k, KT, x_center = 0.0, x_lim = 10, v_lim = 10, num_trials = 10000):
    """
    Calculates the (position, velocity) pairs that correspond to the
    canonical ensemble for a ball (mass=1) on a spring (constant k).
    
    This algorithm implements Metropolis acceptance criteria to perform
    a Monte Carlo simulation.

    Args:
        k (float): The spring constant (unitless). Must be positive.
        KT (float): Temperature factor of the system (inverse of beta; reduced units).
        x_center (float): The center of the spring (default 0.0).
        x_lim (float): limit for search in x.
        v_lim (float): limit for search in v.
        num_trials (int): The number of trials to perform.
        
    Returns:
        pd.DataFrame: A history of all Markov chain and proposed states.
    """

    history_data = {
        'x_proposed': [],
        'v_proposed': [],
        'E_proposed': [],
        'x_chain': [],
        'v_chain': [],
        'E_chain': [],
        'accepted': []
    }

    # --- Initialize chain ---
    x_current, v_current = x_center, 0.0
    E_current = total_energy_physical(x_current, v_current, k, x_center)

    for _ in range(num_trials):

        # --- Propose ---
        x_prop = np.random.uniform(x_center - x_lim, x_center + x_lim)
        v_prop = np.random.uniform(-v_lim, v_lim)
        E_prop = total_energy_physical(x_prop, v_prop, k, x_center)

        # --- Acceptance test ---
        delta_E = E_prop - E_current
        P_acc = acceptance_probability(delta_E, KT)

        accepted = np.random.rand() < P_acc

        if accepted:
            x_current, v_current, E_current = x_prop, v_prop, E_prop

        # --- Log proposal ---
        history_data['x_proposed'].append(x_prop)
        history_data['v_proposed'].append(v_prop)
        history_data['E_proposed'].append(E_prop)

        # --- Log Markov chain state ---
        history_data['x_chain'].append(x_current)
        history_data['v_chain'].append(v_current)
        history_data['E_chain'].append(E_current)
        
        history_data['accepted'].append(accepted)

    return pd.DataFrame(history_data)

In [ ]:
# --- Calculate probability distribution from theory and using Metropolis-Hastings ---
k, KT = 1, 1 # define here to ensure same k and KT used in our calculations
df_theory = calculate_theoretical_distribution(k=k, KT=KT)
df_naive = metropolis_naive(k=k, KT=KT, num_trials=100000)


In [ ]:
# --- Set up a 2x2 grid ---
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# --- Top left: Bivariate distribution ---
# Proposed states (colored by acceptance)
sns.histplot(
    data=df_naive,
    x='x_proposed',
    y='v_proposed',
    hue='accepted',
    ax=axes[0, 0]
)

# Theoretical distribution
sns.kdeplot(
    data=df_theory,
    x='position',
    y='velocity',
    weights='probability',
    fill=False,
    alpha=0.5,
    ax=axes[0, 0],
    cmap='Grays'
)

axes[0, 0].set_title("Proposed States (Accepted vs Rejected)")
axes[0, 0].set_xlabel("Position")
axes[0, 0].set_ylabel("Velocity")

# --- Top right: position histrogram --- 
sns.histplot(
    data=df_naive,
    x='x_chain',
    stat='density',
    ax=axes[0, 1],
    label='Markov chain',
    color=sns.color_palette()[1]
)

sns.histplot(
    data=df_naive,
    x='x_proposed',
    stat='density',
    alpha=0.3,
    ax=axes[0, 1],
    label='Proposals',
    color=sns.color_palette()[0]
)

axes[0, 1].set_title("Position Distribution")
axes[0, 1].set_xlabel("Position")
axes[0, 1].set_ylabel("Probability density")
axes[0, 1].legend()

# --- Bottom left: Energy histogram --- 
sns.histplot(
    data=df_naive,
    x='E_chain',
    stat='density',
    ax=axes[1, 0],
    label='Markov chain',
    color=sns.color_palette()[1]
)

sns.histplot(
    data=df_naive,
    x='E_proposed',
    stat='density',
    alpha=0.3,
    ax=axes[1, 0],
    label='Proposals',
    color=sns.color_palette()[0]
)

axes[1, 0].set_title("Energy Distribution")
axes[1, 0].set_xlabel("Energy")
axes[1, 0].set_ylabel("Probability density")
axes[1, 0].legend()

# --- Bottom right: velocity histogram ---
sns.histplot(
    data=df_naive,
    x='v_chain',
    stat='density',
    ax=axes[1, 1],
    label='Markov chain',
    color=sns.color_palette()[1]
)

sns.histplot(
    data=df_naive,
    x='v_proposed',
    stat='density',
    alpha=0.3,
    ax=axes[1, 1],
    label='Proposals',
    color=sns.color_palette()[0]
)

axes[1, 1].set_title("Velocity Distribution")
axes[1, 1].set_xlabel("Velocity")
axes[1, 1].set_ylabel("Probability density")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

This is great! We are sampling a distribution that is truly approximating the theoretical distribution. But at the same time we are wasting a lot of computing power on moves that we reject. This is in essense the same problem we tried to overcome with the Gillespie algorithm. We would like to generate states that are accepted more frequently. In other words, we would like to **sample** the **important** states more often. Can we think of some ways to achieve this?

One way is to make our simulation a Markov Chain. That is, the proposed state will be generated by performing some operation to the current state. Because the current state was accepted, small perturbations away from the current state are also likely to be accepted.

In [ ]:
def metropolis_importance(k, KT, x_center = 0.0, x_lim = 10, v_lim = 10, step_size=2.0, num_trials = 10000):
    """
    Calculates the (position, velocity) pairs that correspond to 
    the canonical ensemble for a ball (mass=1) on a spring (constant k).
    
    This algorithm implements Metropolis acceptance criteria to perform
    a Monte Carlo simulation.

    Args:
        k (float): The spring constant (unitless). Must be positive.
        KT (float): Temperature factor of the system (inverse of beta; reduced units).
        x_center (float): The center of the spring (default 0.0).
        x_lim (float): limit for search in x.
        v_lim (float): limit for search in v.
        step_size (float): new states will be sampled by nudging current state by
                           a number drawn uniformly from [-step_size, step_size]
        num_trials (int): The number of trials to perform.
        
    Returns:
        pd.DataFrame: A history of all Markovc chain and proposed states.
    """

    # --- Initialize our history to keep track ---
    history_data = {
        'x_proposed': [],
        'v_proposed': [],
        'E_proposed': [],
        'x_chain': [],
        'v_chain': [],
        'E_chain': [],
        'accepted': []
    }

    # --- Initialize chain ---
    x_current, v_current = x_center, 0.0
    E_current = total_energy_physical(x_current, v_current, k, x_center)

    for _ in range(num_trials):

        # --- Local proposal (importance sampling) ---
        x_prop = x_current + np.random.uniform(-step_size, step_size)
        v_prop = v_current + np.random.uniform(-step_size, step_size)

        # --- Energy of proposal ---
        E_prop = total_energy_physical(x_prop, v_prop, k, x_center)

        # --- Metropolis criterion ---
        delta_E = E_prop - E_current
        P_acc = acceptance_probability(delta_E, KT)

        accepted = np.random.rand() < P_acc

        if accepted:
            x_current, v_current, E_current = x_prop, v_prop, E_prop

        # --- Log proposal ---
        history_data['x_proposed'].append(x_prop)
        history_data['v_proposed'].append(v_prop)
        history_data['E_proposed'].append(E_prop)

        # --- Log Markov chain state ---
        history_data['x_chain'].append(x_current)
        history_data['v_chain'].append(v_current)
        history_data['E_chain'].append(E_current)

        history_data['accepted'].append(accepted)

    return pd.DataFrame(history_data)

In [ ]:
# --- Run simulation with importance sampling ---
df_importance = metropolis_importance(k=k, KT=KT, num_trials=100000)

In [ ]:
# --- Set up a 2x2 grid ---
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# --- Top left: Bivariate distribution ---
# Proposed states (colored by acceptance)
sns.histplot(
    data=df_importance,
    x='x_proposed',
    y='v_proposed',
    hue='accepted',
    ax=axes[0, 0]
)

# Theoretical distribution
sns.kdeplot(
    data=df_theory,
    x='position',
    y='velocity',
    weights='probability',
    fill=False,
    alpha=0.5,
    ax=axes[0, 0],
    cmap='Grays'
)

axes[0, 0].set_title("Proposed States (Accepted vs Rejected)")
axes[0, 0].set_xlabel("Position")
axes[0, 0].set_ylabel("Velocity")

# --- Top right: position histrogram --- 
sns.histplot(
    data=df_importance,
    x='x_chain',
    stat='density',
    ax=axes[0, 1],
    label='Markov chain',
    color=sns.color_palette()[1]
)

sns.histplot(
    data=df_importance,
    x='x_proposed',
    stat='density',
    alpha=0.3,
    ax=axes[0, 1],
    label='Proposals',
    color=sns.color_palette()[0]
)

axes[0, 1].set_title("Position Distribution")
axes[0, 1].set_xlabel("Position")
axes[0, 1].set_ylabel("Probability density")
axes[0, 1].legend()

# --- Bottom left: Energy histogram --- 
sns.histplot(
    data=df_importance,
    x='E_chain',
    stat='density',
    ax=axes[1, 0],
    label='Markov chain',
    color=sns.color_palette()[1]
)

sns.histplot(
    data=df_importance,
    x='E_proposed',
    stat='density',
    alpha=0.3,
    ax=axes[1, 0],
    label='Proposals',
    color=sns.color_palette()[0]
)

axes[1, 0].set_title("Energy Distribution")
axes[1, 0].set_xlabel("Energy")
axes[1, 0].set_ylabel("Probability density")
axes[1, 0].legend()

# --- Bottom right: velocity histogram ---
sns.histplot(
    data=df_importance,
    x='v_chain',
    stat='density',
    ax=axes[1, 1],
    label='Markov chain',
    color=sns.color_palette()[1]
)

sns.histplot(
    data=df_importance,
    x='v_proposed',
    stat='density',
    alpha=0.3,
    ax=axes[1, 1],
    label='Proposals',
    color=sns.color_palette()[0]
)

axes[1, 1].set_title("Velocity Distribution")
axes[1, 1].set_xlabel("Velocity")
axes[1, 1].set_ylabel("Probability density")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

Now for our last sanity check, we will write out the accepatance rates of our two methods:

In [ ]:
print("Naive acceptance rate:", df_naive.accepted.mean())
print("Importance acceptance rate:", df_importance.accepted.mean())

**By drawing proposals from a distribution that more closly resembles our expected distribution, we save a significant amount of computational resources by rejecting fewer moves.** As we begin to think about molecular dynamics simulations, we should understand that we are taking this to the **extreme.** In molecular dynamics simulations we "propose" moves by adjusting the positions of atoms using Newton's equations of motion and we "accept" with a probability of 1. We use MD simulations because for complicated systems like proteins or DNA, it has proven challenging (if not impossible) to find a way to propose moves smartly with Monte Carlo; typically any and all proposal moves that sample randomly will get rejected. However, this may not always be the case and cutting-edge research is geared at circumventing the need for MD simulations altogether by sampling states with software the "emulates" MD simulations, see:
https://doi.org/10.1126/science.adv9817

### **Homework for the reader**
1. What happens when step size in importance sampling is too big? Too small?